# Session Viewer

Interactive visualizer for HRT x ETH Zurich Datathon 2026 sessions.

For each session you can see:
- The OHLC candlesticks (plus the close curve)
- The `seen` portion (first half) vs the `unseen` portion (second half, only available for `train`)
- A vertical marker at the half-way close price (the price at which we’d open our position)
- **Semi-transparent vertical lines** on the price chart at every bar that carries ≥ 1 headline.
  Hover over a line to see that bar’s **count and the exact (wrapped) headlines** — all of them, listed.
- A separate **headline-count sub-chart** below the price chart; hovering a count bar shows the same exact headlines for that bar.
- A scrollable table of every headline in the whole session below the chart.

Use the dropdown / slider / buttons at the top to jump around. All controls stay in sync.


In [4]:
from __future__ import annotations

from functools import lru_cache
from pathlib import Path
from textwrap import wrap

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
from plotly.subplots import make_subplots

DATA_DIR = Path('../data/hrt-eth-zurich-datathon-2026/data')
assert DATA_DIR.exists(), f'Data directory not found: {DATA_DIR.resolve()}'
print('Data dir:', DATA_DIR.resolve())

Data dir: /home/codeviser/dev/datathon-HRT-06-2026/data/hrt-eth-zurich-datathon-2026/data


In [5]:
# ---- Data loading ---------------------------------------------------------
# Each dataset kind points at the pair of files (bars + headlines) and whether
# an 'unseen' second half is available. We lazy-load the parquet files and
# cache them so switching between datasets is instant.

DATASETS = {
    'train':        {'bars_seen': 'bars_seen_train.parquet',
                     'bars_unseen': 'bars_unseen_train.parquet',
                     'headlines_seen': 'headlines_seen_train.parquet',
                     'headlines_unseen': 'headlines_unseen_train.parquet'},
    'public_test':  {'bars_seen': 'bars_seen_public_test.parquet',
                     'bars_unseen': None,
                     'headlines_seen': 'headlines_seen_public_test.parquet',
                     'headlines_unseen': None},
    'private_test': {'bars_seen': 'bars_seen_private_test.parquet',
                     'bars_unseen': None,
                     'headlines_seen': 'headlines_seen_private_test.parquet',
                     'headlines_unseen': None},
}

@lru_cache(maxsize=None)
def _load(filename: str) -> pd.DataFrame:
    path = DATA_DIR / filename
    print(f'  loading {filename} ...')
    return pd.read_parquet(path)

def load_dataset(kind: str) -> dict[str, pd.DataFrame | None]:
    spec = DATASETS[kind]
    out: dict[str, pd.DataFrame | None] = {}
    for key, fname in spec.items():
        out[key] = _load(fname) if fname else None
    return out

@lru_cache(maxsize=None)
def sessions_for(kind: str) -> tuple[int, ...]:
    bars = load_dataset(kind)['bars_seen']
    return tuple(sorted(bars['session'].unique().tolist()))

# Prime the cache for train so the first interaction is snappy.
_ = load_dataset('train')
print('Sessions per dataset:')
for k in DATASETS:
    print(f'  {k:>12}: {len(sessions_for(k)):,}')

  loading bars_seen_train.parquet ...
  loading bars_unseen_train.parquet ...
  loading headlines_seen_train.parquet ...
  loading headlines_unseen_train.parquet ...
Sessions per dataset:
         train: 1,000
  loading bars_seen_public_test.parquet ...
  loading headlines_seen_public_test.parquet ...
   public_test: 10,000
  loading bars_seen_private_test.parquet ...
  loading headlines_seen_private_test.parquet ...
  private_test: 10,000


  private_test: 10,000


In [6]:
# ---- Per-session assembly -------------------------------------------------
# Stitch together seen/unseen bars and headlines for a given session.

def session_frames(kind: str, session_id: int):
    d = load_dataset(kind)

    seen_bars = d['bars_seen']
    seen_bars = seen_bars[seen_bars['session'] == session_id].sort_values('bar_ix')

    unseen_bars = d['bars_unseen']
    if unseen_bars is not None:
        unseen_bars = unseen_bars[unseen_bars['session'] == session_id].sort_values('bar_ix')
    else:
        unseen_bars = seen_bars.iloc[0:0]

    all_bars = pd.concat([seen_bars, unseen_bars], ignore_index=True).sort_values('bar_ix')
    all_bars['phase'] = ['seen'] * len(seen_bars) + ['unseen'] * len(unseen_bars)

    headline_parts = []
    for key, phase in (('headlines_seen', 'seen'), ('headlines_unseen', 'unseen')):
        hl = d.get(key)
        if hl is None:
            continue
        sub = hl[hl['session'] == session_id].copy()
        sub['phase'] = phase
        headline_parts.append(sub)
    headlines = (pd.concat(headline_parts, ignore_index=True)
                 if headline_parts else
                 pd.DataFrame(columns=['session', 'headline', 'bar_ix', 'phase']))
    headlines = headlines.sort_values('bar_ix').reset_index(drop=True)

    halfway_bar_ix = int(seen_bars['bar_ix'].max()) if len(seen_bars) else None
    return all_bars, headlines, halfway_bar_ix


In [7]:
# ---- Plotting -------------------------------------------------------------

HEADLINE_WRAP = 72          # chars per line when wrapping a full headline for hover
VLINE_SAMPLES = 24          # sample points per vertical line so hover registers anywhere along it
MAX_HOVER_HEADLINES = 6     # beyond this, truncate hover with a "+N more" footer


def _wrap_hover(text: str) -> str:
    return '<br>'.join(wrap(text, HEADLINE_WRAP)) or str(text)


def _bar_hover_text(bar_ix: int, bar_headlines) -> str:
    """Hover for a bar's vertical line: count + full (wrapped) text of every headline at that bar."""
    n = len(bar_headlines)
    header = f"<b>bar {bar_ix}</b> &middot; {n} headline{'s' if n != 1 else ''}"
    shown = bar_headlines.head(MAX_HOVER_HEADLINES)
    lines = [f"&bull; {_wrap_hover(row['headline'])}" for _, row in shown.iterrows()]
    if n > MAX_HOVER_HEADLINES:
        lines.append(f"<i>&hellip; +{n - MAX_HOVER_HEADLINES} more (see table below)</i>")
    return header + '<br>' + '<br>'.join(lines)


def _count_bar_hover_text(bar_ix: int, bar_headlines) -> str:
    """Hover for the counts-subplot bars: same info as the vertical-line hover."""
    return _bar_hover_text(bar_ix, bar_headlines)


def plot_session(kind: str, session_id: int) -> go.Figure:
    bars, headlines, halfway = session_frames(kind, session_id)
    if bars.empty:
        raise ValueError(f'No bars for session {session_id} in {kind}')

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.04,
        row_heights=[0.78, 0.22],
        subplot_titles=('OHLC (candles) + close line  \u00b7  vertical lines mark headlines',
                        'Headline count per bar  \u00b7  hover for exact headlines'),
    )

    # --- candlestick ---
    fig.add_trace(
        go.Candlestick(
            x=bars['bar_ix'], open=bars['open'], high=bars['high'],
            low=bars['low'], close=bars['close'],
            name='OHLC', increasing_line_color='#26a69a', decreasing_line_color='#ef5350',
            showlegend=False,
        ), row=1, col=1)

    # --- close price line (per-phase) ---
    for phase, color in (('seen', '#1f77b4'), ('unseen', '#9467bd')):
        ph = bars[bars['phase'] == phase]
        if ph.empty:
            continue
        fig.add_trace(
            go.Scatter(x=ph['bar_ix'], y=ph['close'], mode='lines',
                       line=dict(color=color, width=1.6),
                       name=f'close ({phase})',
                       hovertemplate='bar %{x}<br>close %{y:.4f}<extra></extra>'),
            row=1, col=1)

    # --- semi-transparent vertical lines at every bar that has headlines ---
    if not headlines.empty:
        grouped = headlines.groupby('bar_ix')
        phase_by_bar = dict(zip(bars['bar_ix'], bars['phase']))

        # Pre-compute hover text for every bar that has headlines
        hover_by_bar = {int(bar_ix): _bar_hover_text(int(bar_ix), g) for bar_ix, g in grouped}

        # y extent for the vertical lines -- span the full price range plus a small pad
        y_lo = float(bars['low'].min())
        y_hi = float(bars['high'].max())
        pad = (y_hi - y_lo) * 0.03 if y_hi > y_lo else max(abs(y_hi), 1e-6) * 0.03
        y_lo -= pad
        y_hi += pad
        ys_sample = [y_lo + (y_hi - y_lo) * i / (VLINE_SAMPLES - 1)
                     for i in range(VLINE_SAMPLES)]

        # One trace per phase; each vertical line is VLINE_SAMPLES points stacked
        # at the same x, segments separated by None so they render independently.
        series = {'seen':   {'x': [], 'y': [], 'text': []},
                  'unseen': {'x': [], 'y': [], 'text': []}}

        for bar_ix, group in grouped:
            hover = hover_by_bar[int(bar_ix)]
            phase = phase_by_bar.get(bar_ix, 'seen')
            s = series[phase]
            for y in ys_sample:
                s['x'].append(bar_ix)
                s['y'].append(y)
                s['text'].append(hover)
            s['x'].append(None); s['y'].append(None); s['text'].append('')

        for phase, color in (('seen',   'rgba(44,160,44,0.35)'),
                             ('unseen', 'rgba(255,127,14,0.40)')):
            s = series[phase]
            if not s['x']:
                continue
            fig.add_trace(
                go.Scatter(
                    x=s['x'], y=s['y'], mode='lines',
                    line=dict(color=color, width=2),
                    name=f'headline ({phase})',
                    text=s['text'],
                    hovertemplate='%{text}<extra></extra>',
                    hoverlabel=dict(bgcolor='white', font=dict(size=11), align='left'),
                    connectgaps=False,
                ), row=1, col=1)

        # --- headline count bar chart (row 2) --------------------------------
        # Hover on each count-bar also shows exact headlines for that bar.
        counts = (headlines.groupby(['bar_ix', 'phase']).size()
                  .unstack(fill_value=0))
        for phase, color in (('seen', '#2ca02c'), ('unseen', '#ff7f0e')):
            if phase not in counts.columns:
                continue
            sub = counts[counts[phase] > 0]
            xs = list(sub.index)
            ys = [int(v) for v in sub[phase].values]
            texts = [hover_by_bar.get(int(x), f"<b>bar {int(x)}</b>") for x in xs]
            fig.add_trace(
                go.Bar(x=xs, y=ys, name=f'#headlines ({phase})',
                       marker_color=color, opacity=0.85, showlegend=False,
                       text=texts,
                       hovertemplate='%{text}<extra></extra>',
                       hoverlabel=dict(bgcolor='white', font=dict(size=11), align='left')),
                row=2, col=1)
    else:
        fig.add_annotation(text='(no headlines for this session)', showarrow=False,
                           xref='x2 domain', yref='y2 domain', x=0.5, y=0.5,
                           font=dict(color='#888'))

    # --- shade unseen region and mark halfway ---
    if halfway is not None and (bars['phase'] == 'unseen').any():
        fig.add_vrect(x0=halfway + 0.5, x1=bars['bar_ix'].max() + 0.5,
                      fillcolor='#9467bd', opacity=0.07, line_width=0,
                      row='all', col=1)
    halfway_close = None
    if halfway is not None:
        halfway_close = float(bars.loc[bars['bar_ix'] == halfway, 'close'].iloc[0])
        fig.add_vline(x=halfway + 0.5, line=dict(color='#555', width=1, dash='dash'),
                      row='all', col=1)
        fig.add_annotation(x=halfway + 0.5, y=halfway_close,
                           text=f'half-way close = {halfway_close:.4f}',
                           showarrow=True, arrowhead=2, ax=40, ay=-30,
                           row=1, col=1)

    final_close = float(bars['close'].iloc[-1])
    ret = ((final_close / halfway_close - 1) * 100
           if halfway_close is not None and (bars['phase'] == 'unseen').any() else None)
    ret_txt = f" \u2014 return {ret:+.2f}%" if ret is not None else ''

    # Summary line shown above the chart and mirrored in the whole-session hover.
    total_headlines = len(headlines)
    bars_with_news = 0 if headlines.empty else headlines['bar_ix'].nunique()

    fig.update_layout(
        title=(f"{kind} \u2014 session {session_id}  "
               f"({len(bars)} bars, {total_headlines} headlines across "
               f"{bars_with_news} bars){ret_txt}"),
        height=680, xaxis_rangeslider_visible=False,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        margin=dict(l=50, r=20, t=70, b=40),
        hovermode='closest',
    )
    fig.update_xaxes(title_text='bar_ix', row=2, col=1)
    fig.update_yaxes(title_text='price', row=1, col=1)
    fig.update_yaxes(title_text='#headlines', row=2, col=1)
    return fig


In [8]:
# ---- Headlines table helper ----------------------------------------------

def render_headlines_table(kind: str, session_id: int) -> HTML:
    _, headlines, halfway = session_frames(kind, session_id)
    if headlines.empty:
        return HTML('<i>No headlines for this session.</i>')
    df = headlines[['bar_ix', 'phase', 'headline']].copy()
    df.columns = ['bar', 'phase', 'headline']
    styled = (df.style
                .hide(axis='index')
                .set_table_styles([
                    {'selector': 'th', 'props': 'text-align:left; background:#f4f4f4; padding:4px 8px;'},
                    {'selector': 'td', 'props': 'text-align:left; padding:4px 8px; vertical-align:top;'},
                ])
                .apply(lambda s: ['background-color:#fff6e5' if v == 'unseen' else '' for v in s],
                       subset=['phase']))
    return HTML(
        f'<div style="max-height:260px; overflow:auto; border:1px solid #ddd; '
        f'border-radius:6px; padding:4px 8px;">{styled.to_html()}</div>')


In [9]:
# ---- Interactive widget ---------------------------------------------------

dataset_dd = widgets.Dropdown(options=list(DATASETS.keys()), value='train',
                              description='dataset:', style={'description_width': '80px'})
session_dd = widgets.Dropdown(options=sessions_for('train'), value=sessions_for('train')[0],
                              description='session:', style={'description_width': '80px'},
                              layout=widgets.Layout(width='240px'))
session_slider = widgets.IntSlider(min=min(sessions_for('train')), max=max(sessions_for('train')),
                                   value=sessions_for('train')[0], description=' ',
                                   continuous_update=False,
                                   layout=widgets.Layout(width='420px'))
prev_btn = widgets.Button(description='◀ prev', layout=widgets.Layout(width='80px'))
next_btn = widgets.Button(description='next ▶', layout=widgets.Layout(width='80px'))
rand_btn = widgets.Button(description='random', layout=widgets.Layout(width='80px'))

plot_out = widgets.Output()
table_out = widgets.Output()

_sync_guard = {'busy': False}

def _render(kind: str, session_id: int):
    with plot_out:
        plot_out.clear_output(wait=True)
        fig = plot_session(kind, session_id)
        fig.show()
    with table_out:
        table_out.clear_output(wait=True)
        display(render_headlines_table(kind, session_id))

def _refresh_session_options():
    sessions = sessions_for(dataset_dd.value)
    current = session_dd.value
    if current not in sessions:
        current = sessions[0]
    _sync_guard['busy'] = True
    try:
        session_dd.options = sessions
        session_dd.value = current
        session_slider.min = min(sessions)
        session_slider.max = max(sessions)
        session_slider.value = current
    finally:
        _sync_guard['busy'] = False
    _render(dataset_dd.value, current)

def _set_session(new_value: int):
    sessions = sessions_for(dataset_dd.value)
    if new_value not in sessions:
        return
    _sync_guard['busy'] = True
    try:
        session_dd.value = new_value
        session_slider.value = new_value
    finally:
        _sync_guard['busy'] = False
    _render(dataset_dd.value, new_value)

def _on_dataset_change(change):
    if change['name'] != 'value':
        return
    _refresh_session_options()

def _on_session_dd_change(change):
    if change['name'] != 'value' or _sync_guard['busy']:
        return
    _set_session(change['new'])

def _on_slider_change(change):
    if change['name'] != 'value' or _sync_guard['busy']:
        return
    # snap to nearest available session
    sessions = sessions_for(dataset_dd.value)
    target = min(sessions, key=lambda s: abs(s - change['new']))
    _set_session(target)

def _step(delta: int):
    sessions = sessions_for(dataset_dd.value)
    i = sessions.index(session_dd.value)
    j = (i + delta) % len(sessions)
    _set_session(sessions[j])

import random
prev_btn.on_click(lambda _: _step(-1))
next_btn.on_click(lambda _: _step(+1))
rand_btn.on_click(lambda _: _set_session(random.choice(sessions_for(dataset_dd.value))))

dataset_dd.observe(_on_dataset_change, names='value')
session_dd.observe(_on_session_dd_change, names='value')
session_slider.observe(_on_slider_change, names='value')

controls = widgets.VBox([
    widgets.HBox([dataset_dd, session_dd, prev_btn, next_btn, rand_btn]),
    widgets.HBox([widgets.Label('slider:'), session_slider]),
])

display(controls, plot_out, widgets.HTML('<h4 style="margin:8px 0 4px 0;">Headlines</h4>'), table_out)
_render(dataset_dd.value, session_dd.value)

Output()

HTML(value='<h4 style="margin:8px 0 4px 0;">Headlines</h4>')

Output()

## Static snapshots

Below are a few non-interactive snapshots so the notebook is useful even when
rendered without a live kernel (e.g. on GitHub). Each of these runs the same
plotting code used by the widget above.

In [7]:
# A training session (has unseen half + headlines)
plot_session('train', 0).show()

In [8]:
# A public-test session (only seen half)
plot_session('public_test', 1000).show()

In [9]:
# A private-test session
plot_session('private_test', 11000).show()